Here is some updating of prompts based on prior publications, further knowledge, and just like, generally growing as people.

In [1]:
orig_dir = '/Users/jnaiman/Dropbox/Paper_iConf2025/data/visual_qa_multipanel_orig/data_full_v2/'

In [2]:
import json
from glob import glob
import numpy as np

# debug
from importlib import reload

from sys import path
path.append('../')
from utils.plot_qa_utils import init_qa_pairs

from utils.plot_parameters import plot_types_params
plot_params = plot_types_params.copy()

import utils.plot_qa_utils
reload(utils.plot_qa_utils)

verbose_qa = True

In [3]:
#stats = {'minimum':np.min, 'maximum':np.max, 'median':np.median, 'mean':np.mean}
stats = {'median':np.median, 'mean':np.mean}

linestyles = ['-', '--', ':'] # only use a subset of the linestyles

In [4]:
# calculate plottypes

# first, use probabilities to choose
# Prob for getting line
plot_params['line']['prob'] = 1
plot_params['histogram']['prob'] = 1
plot_params['scatter']['prob'] = 1
plot_params['contour']['prob'] = 0 # didn't do for this one

plot_types = []
for k,v in plot_params.items():
    if v['prob'] > 0:
        plot_types.append(k)

plot_types

['line', 'histogram', 'scatter']

In [5]:
files_json_qa = glob(orig_dir + '*_qa.json*')
files_json_qa[:3]

['/Users/jnaiman/Dropbox/Paper_iConf2025/data/visual_qa_multipanel_orig/data_full_v2/Picture87_qa.json',
 '/Users/jnaiman/Dropbox/Paper_iConf2025/data/visual_qa_multipanel_orig/data_full_v2/Picture97_qa.json',
 '/Users/jnaiman/Dropbox/Paper_iConf2025/data/visual_qa_multipanel_orig/data_full_v2/Picture168_qa.json']

In [ ]:
import utils.figure_level_qa_utils
reload(utils.figure_level_qa_utils)
from utils.figure_level_qa_utils import figure_qa_how_many_panels, figure_qa_plotting_style, figure_qa_colormap, figure_qa_aspect_ratio, q_plot_titles, q_plot_axis_labels, q_ticklabels, q_plot_types

import utils.general_plot_level_qa_utils
reload(utils.general_plot_level_qa_utils)
from utils.general_plot_level_qa_utils import q_errorbars_existance_lines

def figure_level_qa(data, qa_pairs, plot_types, verbose=False):
    ######## FIGURE LEVEL, L1 ########
    qa_pairs = figure_qa_how_many_panels(data, qa_pairs, verbose=verbose)
    qa_pairs = figure_qa_plotting_style(data, qa_pairs, verbose=verbose)
    qa_pairs = figure_qa_colormap(data, qa_pairs, verbose=verbose)
    qa_pairs = figure_qa_aspect_ratio(data, qa_pairs, verbose=verbose)
    # basic, plot level in general
    qa_pairs = q_plot_titles(data, qa_pairs, verbose=verbose)
    qa_pairs = q_plot_axis_labels(data, qa_pairs, axis='x', verbose=verbose)
    qa_pairs = q_plot_axis_labels(data, qa_pairs, axis='y', verbose=verbose)
    qa_pairs = q_ticklabels(data, qa_pairs, axis='x', verbose=verbose)
    qa_pairs = q_ticklabels(data, qa_pairs, axis='y', verbose=verbose)
    qa_pairs = q_plot_types(data, qa_pairs, plot_types, use_list=True, verbose=verbose)
    qa_pairs = q_plot_types(data, qa_pairs, plot_types, use_list=False, verbose=verbose)

    return qa_pairs

In [16]:
import utils.linear_plot_qa_utils
reload(utils.linear_plot_qa_utils)

from utils.linear_plot_qa_utils import q_nlines_plot_plotnums, q_stats_lines, \
    q_relationship_lines, q_linestyles_lines

def plot_level_line_qa(data, qa_pairs, iplot, stats, verbose_qa=False):
    ######### L1 #########
    qa_pairs = q_nlines_plot_plotnums(data, qa_pairs, 
                                        plot_num = iplot, 
                                        verbose=verbose_qa)
    # note: line colors don't work right now
    # qa_pairs = q_colors_lines(data, qa_pairs, 
    #                                   plot_num = iplot, 
    #                                   verbose=verbose_qa)    
    qa_pairs = q_linestyles_lines(data, qa_pairs, plot_num=iplot, 
                                    use_words=True, use_list = True,
                        linestyle_list=linestyles, verbose=verbose_qa)      
    ######### L2 #########
    # stats items
    for k,v in stats.items(): # for all stats
        qa_pairs = q_stats_lines(data, qa_pairs, stat={k:v}, 
                                    plot_num=iplot, use_words=True, 
                                    verbose=verbose_qa)
    ######## L3 #########
    for use_list in [True,False]:
        for use_nlines in [True, False]:
            qa_pairs = q_relationship_lines(data, qa_pairs, plot_num=iplot, use_words=True, 
                                        use_nlines = use_nlines, 
                                        verbose=verbose_qa, use_list=use_list)
            
    return qa_pairs

In [ ]:
for json_path in files_json_qa:
    with open(json_path, 'r') as f:
        qa = json.load(f)
        qa = json.loads(qa)

    # also full json
    json_path_full = json_path.replace('_qa.json', '_fullData.json')
    with open(json_path_full,'r') as f:
        data = json.load(f)
        data = json.loads(data)

    # new
    qa_pairs = init_qa_pairs()

    ##################################
    ########## L1 ###################
    #################################

    ######## FIGURE LEVEL ########
    qa_pairs = figure_level_qa(data, qa_pairs, plot_types, verbose=verbose_qa)

    ###### LOOP AND PLOT-LEVEL ######
    plot_nums = []
    for k,v in data.items():
        if 'plot' in k:
            plot_nums.append(int(k.split('plot')[-1]))

    if verbose_qa:
        print('')
        print('----------- PLOT SPECIFIC ----------')
        print('')
    for iplot in plot_nums:
        # general plot-level questions
        ##### L2 ######
        for axis in ['x', 'y']:
            qa_pairs = q_errorbars_existance_lines(data, qa_pairs, 
                                              plot_num = iplot, 
                                              verbose=verbose_qa, 
                                              axis=axis)

        if data['plot'+str(iplot)]['type'] == 'line': # line plots
            qa_pairs = plot_level_line_qa(data, qa_pairs, iplot, stats, verbose_qa=verbose_qa)
        elif data['plot'+str(iplot)]['type'] == 'scatter':
            ## number of scatters
            ## number of ticklines for colorbar??
            ## mean/median for x/y
            ## mean/median for color


            import sys; sys.exit()

QUESTION: How many panels are in this figure?
ANSWER: {'nrows': 4, 'ncols': 2}

QUESTION: What is the plot style used in this figure?
ANSWER: {'plot style': '_classic_test_patch'}

QUESTION: What is the colormap that was used in this figure?
ANSWER: {'colormap': 'gray'}

QUESTION: What is the aspect ratio of this figure?
ANSWER: {'aspect ratio': 2.6199889122138043}

QUESTION: What are the titles for each figure panel?
ANSWER: {'title': ['', 'Shape', 'LOOK ANALYSES ROLE ${\\bf a}_i$ ', 'OPTICS SERVICE SOURCE BASE', '', '', 'PSPC SUM', 'Globular Couple Crust']}

QUESTION: You are a helpful assistant that can analyze images.  What are the x-axis titles for each figure panel? Please format the output as a json as {"xlabels":[]}, where the list is a list of strings of all of the x-axis titles. If there is a single plot, this should be one element in this list, and if there are multiple plots the list should be in row-major (C-style) order. If a plot does not have an x-axis title, then denot

SystemExit: 

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [19]:
qa_pairs['Level 1']['Figure-level questions']['plot types']

{'Q': 'You are a helpful assistant that can analyze images.  What are the plot types for each panel in the figure? Please format the output as a json as {"plot types":[]}, where each element of the list refers to the plot type of a single panel. If there is a single plot, this should be one element in this list, and if there are multiple plots the list should be in row-major (C-style) order. ',
 'A': {'plot types': ['line',
   'line',
   'line',
   'line',
   'scatter',
   'scatter',
   'scatter',
   'line']},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'What are the plot types for each panel in the figure?',
 'format': 'Please format the output as a json as {"plot types":[]}, where each element of the list refers to the plot type of a single panel. If there is a single plot, this should be one element in this list, and if there are multiple plots the list should be in row-major (C-style) order. '}

In [ ]:
qa_pairs['Level 1']['Figure-level questions']['plot types (list)']

{'Q': 'You are a helpful assistant that can analyze images. Please choose each plot type from the following list: [line, histogram, scatter]. What are the plot types for each panel in the figure? Please format the output as a json as {"plot types":[]}, where each element of the list refers to the plot type of a single panel. If there is a single plot, this should be one element in this list, and if there are multiple plots the list should be in row-major (C-style) order. ',
 'A': {'plot types': ['line',
   'line',
   'line',
   'line',
   'scatter',
   'scatter',
   'scatter',
   'line']},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': 'Please choose each plot type from the following list: [line, histogram, scatter].',
 'question': 'What are the plot types for each panel in the figure?',
 'format': 'Please format the output as a json as {"plot types":[]}, where each element of the list refers to the plot type of a single panel. If there is a single plot

In [ ]:
qa_new

{'Level 1': {'Figure-level questions': {}, 'Plot-level questions': {}},
 'Level 2': {'Plot-level questions': {}},
 'Level 3': {'Plot-level questions': {}}}